# Drive Diagnostic — Where did nb01b's downloads go?

Standalone diagnostic notebook. **Just run the single cell below** — no edits needed.

This finds your BSISO_SSL_Project folder (handles upper/lowercase), lists every `.nc` and `X_*.npy` file with size + modtime, and decides whether nb03 (preprocessing) needs to be rerun.

Send the output back to chat.

---

In [1]:
import os, glob, datetime

# 1. Mount Drive if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# 2. Find the project folder — try common casings
candidates = ['BSISO_SSL_Project', 'BSISO_SSL_project', 'BSISO_ssl_project', 'bsiso_ssl_project']
PROJECT_DIR = None
for name in candidates:
    p = f'/content/drive/MyDrive/{name}'
    if os.path.exists(p):
        PROJECT_DIR = p
        print(f'✓ Found project root: {p}')
        break

if PROJECT_DIR is None:
    print('✗ No project folder found. Top-level Drive contents (first 30):')
    try:
        items = sorted(os.listdir('/content/drive/MyDrive'))
        for it in items[:30]:
            print(f'  {it}')
    except Exception as e:
        print(f'  (could not list: {e})')
else:
    # 3. Recursively find all .nc files
    print(f'\n--- All .nc files under {os.path.basename(PROJECT_DIR)} ---')
    nc_count = 0
    nc_files_found = []
    for root, _, files in os.walk(PROJECT_DIR):
        for f in sorted(files):
            if f.endswith('.nc'):
                p = os.path.join(root, f)
                mt = datetime.datetime.fromtimestamp(os.path.getmtime(p))
                sz_mb = os.path.getsize(p) / 1e6
                rel = os.path.relpath(p, PROJECT_DIR)
                print(f'  {mt.strftime("%Y-%m-%d %H:%M")}  {sz_mb:7.1f} MB   {rel}')
                nc_count += 1
                nc_files_found.append(p)
    print(f'(Total: {nc_count} .nc files)')

    # 4. Recursively find all X_*.npy files
    print(f'\n--- All X_*.npy files under {os.path.basename(PROJECT_DIR)} ---')
    npy_count = 0
    npy_files_found = []
    for root, _, files in os.walk(PROJECT_DIR):
        for f in sorted(files):
            if f.startswith('X_') and f.endswith('.npy'):
                p = os.path.join(root, f)
                mt = datetime.datetime.fromtimestamp(os.path.getmtime(p))
                sz_mb = os.path.getsize(p) / 1e6
                rel = os.path.relpath(p, PROJECT_DIR)
                print(f'  {mt.strftime("%Y-%m-%d %H:%M")}  {sz_mb:7.1f} MB   {rel}')
                npy_count += 1
                npy_files_found.append(p)
    print(f'(Total: {npy_count} X_*.npy files)')

    # 5. Decision: do .npy files need refreshing?
    print('\n--- Decision logic ---')
    raw_mjjas_nc = [f for f in nc_files_found if 'MJJAS' in f or 'mjjas' in f]
    processed_npy = [f for f in npy_files_found if '/processed/' in f.replace('\\', '/')]

    if not raw_mjjas_nc and not processed_npy:
        print(f'  ✗ No raw MJJAS .nc AND no processed X_*.npy → fresh setup needed')
        print(f'  → Re-run nb01b to download, then nb03 to preprocess.')
    elif raw_mjjas_nc and not processed_npy:
        print(f'  ⚠ Have {len(raw_mjjas_nc)} raw MJJAS .nc but NO processed X_*.npy')
        print(f'  → Run nb03 to create them.')
    elif not raw_mjjas_nc and processed_npy:
        print(f'  ⚠ Have {len(processed_npy)} processed X_*.npy but NO raw MJJAS .nc')
        print(f'  → Strange — processed files exist but raw download missing. Where is RAW_DIR pointing?')
    else:
        latest_raw = max(os.path.getmtime(f) for f in raw_mjjas_nc)
        latest_npy = max(os.path.getmtime(f) for f in processed_npy)
        delta_hours = (latest_raw - latest_npy) / 3600
        if delta_hours > 1:
            print(f'  ⚠ Latest raw MJJAS .nc is {delta_hours:.1f} h NEWER than latest processed X_*.npy')
            print(f'  → RERUN nb03 (preprocessing). Then nb03b, nb08 (for lp25), then NSV nb17b → nb18c → nb19 → nb20.')
        else:
            print(f'  ✓ Processed X_*.npy files are up-to-date with raw .nc files.')
            print(f'  → No rerun needed.')

    # 6. Useful one-liner summary at the bottom
    print(f'\n--- Summary ---')
    print(f'  PROJECT_DIR  = {PROJECT_DIR}')
    print(f'  .nc count    = {nc_count}')
    print(f'  X_*.npy count = {npy_count}')

Mounted at /content/drive
✓ Found project root: /content/drive/MyDrive/BSISO_SSL_Project

--- All .nc files under BSISO_SSL_Project ---
  2026-06-09 18:30     43.6 MB   data/raw/OLR_MJJAS_1979_2023.nc
  2026-06-06 21:28     21.3 MB   data/raw/u850_v850_MJJAS_1979_1989.nc
  2026-06-06 21:51     19.4 MB   data/raw/u850_v850_MJJAS_1990_1999.nc
  2026-06-06 22:13     19.4 MB   data/raw/u850_v850_MJJAS_2000_2009.nc
  2026-06-06 22:37     19.4 MB   data/raw/u850_v850_MJJAS_2010_2019.nc
  2026-06-06 22:46      7.8 MB   data/raw/u850_v850_MJJAS_2020_2023.nc
  2026-03-03 03:34      4.8 MB   data/raw/_snapshot12z_backup/OLR_July_1979_2023.nc
  2026-04-04 20:17     28.4 MB   data/raw/_snapshot12z_backup/OLR_MJJAS_1979_2023.nc
  2026-05-02 17:52     19.3 MB   data/raw/_snapshot12z_backup/precip_MJJAS_1979_2023.nc
  2026-03-03 03:24      2.9 MB   data/raw/_snapshot12z_backup/u850_v850_July_1979_1989.nc
  2026-03-03 03:25      2.7 MB   data/raw/_snapshot12z_backup/u850_v850_July_1990_1999.nc
  2026-

---
**Send the output back to chat** and I'll tell you the precise rerun sequence (or whether you can skip it entirely).

If you see `✗ No project folder found`, the project might be saved under a different name on Drive — look at the listed Drive contents and tell me what's there.
